# Stream A — Improved Pipeline
## Whisper → Pyannote → OpenSMILE (eGeMAPSv02)

| Step | Tool | Purpose |
|------|------|---------|
| 0 | Config | Paths, tokens, parameters |
| 1 | Environment check | Verify all dependencies |
| 2 | Subject discovery | Scan data directory |
| 3 | Whisper | Transcription + timestamps |
| 4 | Pyannote | Speaker diarization — isolate participant only |
| 5 | Segment export | Export participant-only WAV segments |
| 6 | OpenSMILE | Extract eGeMAPSv02 (88 features) per segment |
| 7 | Aggregation | mean/std/p10/p50/p90 → one row per subject |
| 8 | Merge labels | Join PHQ-8 scores from train/dev split CSV |
| 9 | Output + QC | Save CSV, validate, plot feature distributions |

> **Key improvement over v1**: Pyannote filters out Ellie (interviewer) before feature extraction,
> so features reflect participant speech only. eGeMAPSv02 is the academic standard for
> depression/emotion recognition — directly comparable to published results.

---
## Cell 0 — Configuration

In [ ]:
import os
from pathlib import Path

# ============================================================
# ✏️  Edit this section only
# ============================================================
RAW_DATA_DIR  = Path("/Users/Meihui/Downloads/sync/work/whisperx")
LABELS_DIR    = RAW_DATA_DIR
OUTPUT_DIR    = Path("outputs/features")
SEGMENT_DIR   = Path("outputs/segments")
LOG_DIR       = Path("outputs/logs")

# Load HuggingFace token from environment variable — never hardcode tokens in a repo
# Before running: export HF_TOKEN=hf_...
HF_TOKEN = os.environ.get("HF_TOKEN", "")
assert HF_TOKEN, "Set HF_TOKEN env var before running: export HF_TOKEN=hf_..."

WHISPER_MODEL_SIZE = "base"
PYANNOTE_MODEL    = "pyannote/speaker-diarization-3.1"
TARGET_SR           = 16000
MIN_SEGMENT_DUR_SEC = 0.5
OUTPUT_CSV = OUTPUT_DIR / "improved_features.csv"
# ============================================================
for d in [OUTPUT_DIR, SEGMENT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"Data dir: {RAW_DATA_DIR}")
print(f"Output CSV: {OUTPUT_CSV}")
print("HF_TOKEN: loaded from env ✅")

---
## Cell 1 — Environment Check

In [ ]:
import sys, json, time, logging, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import opensmile
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"OpenSMILE   : {opensmile.__version__}")
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device      : {DEVICE}")
smile = opensmile.Smile(feature_set=opensmile.FeatureSet.eGeMAPSv02, feature_level=opensmile.FeatureLevel.Functionals)
print(f"eGeMAPSv02 features : {len(smile.feature_names)}")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_DIR / f"run_{RUN_ID}.log"), logging.StreamHandler(sys.stdout)])
logger = logging.getLogger("improved")
print("\n✅ Environment OK")

---
## Cell 2 — Subject Discovery

In [ ]:
def discover_subjects(raw_dir):
    subjects = []
    for d in sorted(raw_dir.iterdir()):
        if not d.is_dir(): continue
        session_id = d.name.replace('_P', '')
        audio = d / f'{session_id}_AUDIO.wav'
        if not audio.exists():
            wavs = sorted(d.glob('*.wav'))
            if not wavs: continue
            audio = wavs[0]
        transcript = d / f'{session_id}_TRANSCRIPT.csv'
        subjects.append({'subject_id': d.name, 'session_id': session_id,
                          'audio_path': audio, 'transcript': transcript if transcript.exists() else None})
        t_status = 'transcript ✅' if transcript.exists() else 'transcript ❌'
        print(f'  [{d.name}]  {audio.name}  |  {t_status}')
    return subjects

SUBJECTS = discover_subjects(RAW_DATA_DIR)
print(f'\nTotal: {len(SUBJECTS)} subjects')

---
## Cell 3 — Whisper Transcription
Used as fallback for subjects without official transcripts.
For subjects WITH `XXX_TRANSCRIPT.csv`, the official one is used (more accurate).

In [ ]:
def load_official_transcript(transcript_path):
    """Load DAIC-WOZ official transcript, return participant-only segments."""
    try:
        df = pd.read_csv(transcript_path, sep='\t', names=['start','end','speaker','text'])
        participant = df[df['speaker'] == 'Participant'].copy()
        participant = participant[~participant['text'].str.contains('scrubbed_entry', na=False)]
        return [{'start': float(row.start), 'end': float(row.end),
                 'text': str(row.text), 'speaker': 'PARTICIPANT', 'source': 'official_transcript'}
                for _, row in participant.iterrows()]
    except Exception as e:
        logger.warning(f'Failed to load transcript {transcript_path}: {e}')
        return []

def transcribe_whisper(audio_path, model_size):
    """Fallback: transcribe with faster-whisper."""
    from faster_whisper import WhisperModel
    model = WhisperModel(model_size, device='cpu', compute_type='int8')
    segs, _ = model.transcribe(str(audio_path), beam_size=5, language='en', vad_filter=True)
    return [{'start': s.start, 'end': s.end, 'text': s.text.strip(), 'speaker': 'UNKNOWN', 'source': 'whisper'} for s in segs]

ALL_TRANSCRIPTS = {}
for subj in tqdm(SUBJECTS, desc='Transcription'):
    sid = subj['subject_id']
    cache = LOG_DIR / f'{sid}_transcript.json'
    if cache.exists():
        segments = json.load(open(cache))
    elif subj['transcript'] is not None:
        # Use official DAIC-WOZ transcript (already filtered to participant)
        segments = load_official_transcript(subj['transcript'])
        json.dump(segments, open(cache, 'w'), indent=2)
    else:
        # Fallback to Whisper
        segments = transcribe_whisper(subj['audio_path'], WHISPER_MODEL_SIZE)
        json.dump(segments, open(cache, 'w'), indent=2)
    ALL_TRANSCRIPTS[sid] = segments
    logger.info(f'{sid}: {len(segments)} segments')

---
## Cell 4 — Pyannote Speaker Diarization
Identifies participant vs Ellie. For subjects with official transcripts, used as cross-check.
For subjects without transcripts, this is the primary speaker filter.

**Note**: If all subjects have official transcripts, you can skip this cell
and set `ALL_PARTICIPANT_SEGMENTS = ALL_TRANSCRIPTS.copy()`

In [ ]:
from pyannote.audio import Pipeline as PyannotePipeline

def load_pyannote(hf_token, model_name):
    pipeline = PyannotePipeline.from_pretrained(model_name, use_auth_token=hf_token)
    pipeline.to(torch.device('cpu'))  # Run on CPU to avoid MPS instability
    return pipeline

def identify_participant_speaker(diar_segs):
    """Participant typically speaks more total time than Ellie — pick longest speaker."""
    durations = {}
    for s in diar_segs:
        durations[s['speaker']] = durations.get(s['speaker'], 0) + s['end'] - s['start']
    return max(durations, key=durations.get) if durations else None

pyannote_pipeline = load_pyannote(HF_TOKEN, PYANNOTE_MODEL)
ALL_PARTICIPANT_SEGMENTS = {}

for subj in tqdm(SUBJECTS, desc='Diarization'):
    sid = subj['subject_id']
    cache = LOG_DIR / f'{sid}_diarization.json'
    if cache.exists():
        participant_segs = json.load(open(cache))
    else:
        try:
            diar = pyannote_pipeline(str(subj['audio_path']))
            diar_segs = [{'start': t.start, 'end': t.end, 'speaker': spk} for t, _, spk in diar.itertracks(yield_label=True)]
            p_label = identify_participant_speaker(diar_segs)
            participant_segs = [s for s in diar_segs if s['speaker'] == p_label and s['end']-s['start'] >= MIN_SEGMENT_DUR_SEC]
            json.dump(participant_segs, open(cache, 'w'), indent=2)
        except Exception as e:
            logger.error(f'{sid}: diarization failed — {e}')
            participant_segs = ALL_TRANSCRIPTS.get(sid, [])  # Fallback to transcript
    ALL_PARTICIPANT_SEGMENTS[sid] = participant_segs
    logger.info(f'{sid}: {len(participant_segs)} participant segments')

---
## Cell 5 — Export Participant WAV Segments

In [ ]:
def export_segments(audio_path, segments, out_dir, target_sr=TARGET_SR):
    out_dir.mkdir(parents=True, exist_ok=True)
    audio, _ = librosa.load(str(audio_path), sr=target_sr, mono=True)
    # Peak normalize to remove microphone gain differences
    peak = np.max(np.abs(audio))
    if peak > 1e-6:
        audio = audio * (0.95 / peak)
    valid = []
    for i, seg in enumerate(segments):
        start = float(seg.get('start', 0))
        end   = float(seg.get('end', 0))
        dur   = end - start
        if dur < MIN_SEGMENT_DUR_SEC: continue
        s, e = int(start * target_sr), min(int(end * target_sr), len(audio))
        chunk = audio[s:e]
        if len(chunk) < int(MIN_SEGMENT_DUR_SEC * target_sr): continue
        wav_path = out_dir / f'seg_{i:04d}.wav'
        sf.write(str(wav_path), chunk, target_sr)
        valid.append({**seg, 'wav_path': str(wav_path), 'duration': dur})
    return valid

ALL_EXPORTED_SEGMENTS = {}
for subj in tqdm(SUBJECTS, desc='Exporting segments'):
    sid = subj['subject_id']
    segs = ALL_PARTICIPANT_SEGMENTS.get(sid, [])
    exported = export_segments(subj['audio_path'], segs, SEGMENT_DIR / sid)
    ALL_EXPORTED_SEGMENTS[sid] = exported
    logger.info(f'{sid}: exported {len(exported)} segments')

---
## Cell 6 — OpenSMILE Feature Extraction (eGeMAPSv02)
eGeMAPSv02 = 88 features: F0, loudness, spectral, MFCC 1-4, jitter, shimmer, HNR.
Academic standard for depression/emotion recognition research.

In [ ]:
smile = opensmile.Smile(feature_set=opensmile.FeatureSet.eGeMAPSv02, feature_level=opensmile.FeatureLevel.Functionals)
print(f'eGeMAPSv02: {len(smile.feature_names)} features')

def extract_egemaps(wav_path):
    """Extract eGeMAPSv02 features from a single WAV segment. Returns flat dict of 88 features."""
    return smile.process_file(wav_path).iloc[0].to_dict()

EGEMAPS_FEATURES = {}
for sid, segments in tqdm(ALL_EXPORTED_SEGMENTS.items(), desc='eGeMAPSv02'):
    feats_list = []
    for seg in tqdm(segments, desc=f'  {sid}', leave=False):
        try:
            feats = extract_egemaps(seg['wav_path'])
            feats['seg_start'] = seg['start']
            feats['seg_end']   = seg['end']
            feats['seg_duration'] = seg['duration']
            feats_list.append(feats)
        except Exception as e:
            logger.warning(f'{sid} seg {Path(seg["wav_path"]).name}: {e}')
    EGEMAPS_FEATURES[sid] = feats_list
    logger.info(f'{sid}: eGeMAPSv02 extracted for {len(feats_list)} segments')

---
## Cell 7 — Subject-level Aggregation

In [ ]:
PERCENTILES = [10, 50, 90]

def aggregate(sid, feats):
    """Aggregate segment-level features into one subject row. Stats: mean/std/p10/p50/p90."""
    row = {'subject_id': sid}
    if not feats: return row
    numeric_keys = [k for k in feats[0] if isinstance(feats[0][k], (int, float)) and k not in ('seg_start','seg_end')]
    mat = np.array([[f.get(k, np.nan) for k in numeric_keys] for f in feats], dtype=float)
    for j, key in enumerate(numeric_keys):
        col = mat[:, j]; col = col[~np.isnan(col)]
        if len(col) == 0:
            for stat in ['mean','std'] + [f'p{p}' for p in PERCENTILES]: row[f'{key}_{stat}'] = np.nan
            continue
        row[f'{key}_mean'] = float(np.mean(col))
        row[f'{key}_std']  = float(np.std(col))
        for p in PERCENTILES: row[f'{key}_p{p}'] = float(np.percentile(col, p))
    row['n_segments']       = len(feats)
    row['total_speech_sec'] = float(sum(f.get('seg_duration', 0) for f in feats))
    return row

rows = [aggregate(s['subject_id'], EGEMAPS_FEATURES.get(s['subject_id'], [])) for s in SUBJECTS]
FEATURE_DF = pd.DataFrame(rows)
print(f'Aggregation complete: {FEATURE_DF.shape[0]} subjects x {FEATURE_DF.shape[1]} features')
FEATURE_DF[['subject_id', 'n_segments', 'total_speech_sec']]

---
## Cell 8 — Merge PHQ-8 Labels

In [ ]:
def load_labels(labels_dir):
    """Load and combine train + dev split label CSVs. Returns DataFrame with PHQ8_Binary, PHQ8_Score, Gender."""
    dfs = []
    for fname in ['train_split_Depression_AVEC2017.csv', 'dev_split_Depression_AVEC2017.csv']:
        fpath = labels_dir / fname
        if fpath.exists():
            df = pd.read_csv(fpath); df['split'] = fname.split('_')[0]; dfs.append(df)
            print(f'  Loaded {fname}: {len(df)} subjects')
        else:
            print(f'  NOT FOUND: {fpath}')
    if not dfs: return pd.DataFrame()
    labels = pd.concat(dfs, ignore_index=True)
    labels['subject_id'] = labels['Participant_ID'].astype(str) + '_P'
    return labels[['subject_id', 'PHQ8_Binary', 'PHQ8_Score', 'Gender', 'split']]

LABELS_DF = load_labels(LABELS_DIR)
if not LABELS_DF.empty:
    FEATURE_DF = FEATURE_DF.merge(LABELS_DF, on='subject_id', how='left')
    print(f'After label merge: {FEATURE_DF.shape}')
    print(f"  With PHQ8: {FEATURE_DF['PHQ8_Score'].notna().sum()} subjects")
else:
    print('Label files not found — place train/dev CSVs in:', LABELS_DIR)

---
## Cell 9 — Output + Validation + QC Plot

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

FEATURE_DF.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV.resolve()}  |  Shape: {FEATURE_DF.shape}')

qc_features = [
    ('F0semitoneFrom27.5Hz_sma3nz_amean_mean',     'F0 Mean (semitone)'),
    ('F0semitoneFrom27.5Hz_sma3nz_stddevNorm_mean', 'F0 Std Norm  ← pitch variability'),
    ('loudness_sma3_amean_mean',                    'Loudness Mean'),
    ('loudness_sma3_stddevNorm_mean',               'Loudness Std Norm'),
    ('HNRdBACF_sma3nz_amean_mean',                  'HNR (voice quality)'),
    ('jitterLocal_sma3nz_amean_mean',               'Jitter (voice irregularity)'),
]
available_qc = [(col, label) for col, label in qc_features if col in FEATURE_DF.columns]
if available_qc:
    subjects = FEATURE_DF['subject_id'].tolist()
    x = np.arange(len(subjects))
    fig, axes = plt.subplots(len(available_qc), 1, figsize=(14, len(available_qc) * 2.5))
    if len(available_qc) == 1: axes = [axes]
    colors = plt.cm.tab10.colors
    for i, (col, label) in enumerate(available_qc):
        vals = FEATURE_DF[col].values.astype(float)
        gm, gs = np.nanmean(vals), np.nanstd(vals)
        bc = ['tomato' if abs(v-gm) > 2*gs else colors[i%len(colors)] for v in vals]
        axes[i].bar(x, vals, color=bc, alpha=0.8, edgecolor='white')
        axes[i].axhline(gm, color='black', linestyle='--', linewidth=1, label=f'mean={gm:.3f}')
        axes[i].set_title(label, fontsize=10, fontweight='bold', loc='left')
        axes[i].set_xticks(x); axes[i].set_xticklabels(subjects, rotation=35, ha='right', fontsize=8)
        axes[i].legend(fontsize=8)
    plt.suptitle('QC: eGeMAPSv02 Key Features | Red = >2 SD', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'qc_egemaps_features.png', dpi=150, bbox_inches='tight')
    plt.close()
print('Pipeline complete.')